# Predicting Electric Vehicle Purchases — End-to-End Training

**Competition:** [Kaggle Playground Series S6E9](https://www.kaggle.com/competitions/playground-series-s6e9)  
**Metric:** ROC-AUC  
**Model:** LightGBM, 5-fold stratified CV, native categorical handling  
**Author:** kaustubh (build by opencode)

## What this notebook does

1. Loads `train.csv` and `test.csv` from the Kaggle competition input.
2. Engineers 6 features on top of the 13 raw features (see `src/features.py`).
3. Builds stratified 5-fold CV splits (seed=42).
4. Trains a LightGBM model per fold with native categorical handling, computes OOF predictions and per-fold AUCs.
5. Averages test predictions across folds.
6. Logs params, metrics, OOF array, and test predictions to MLflow (file backend at `/kaggle/working/mlruns/`).
7. Writes a Kaggle-format `submission.csv` to `/kaggle/working/submission.csv`.

## How to run on Kaggle

1. Create a new notebook on Kaggle.
2. Add the competition dataset (`playground-series-s6e9`).
3. Upload the `src/` folder as a Kaggle Dataset, or copy-paste the contents inline.
4. Copy-paste the cells below into the notebook.
5. Run all cells. The submission appears in `/kaggle/working/submission.csv`.
6. The MLflow runs are at `/kaggle/working/mlruns/`. To download, zip them and add as a notebook output:

   ```python
   !cd /kaggle/working && zip -r mlruns.zip mlruns/
   ```

## Expected runtime

- Local CPU on full data: ~5-8 min.
- Kaggle free CPU notebook: ~5-10 min.
- Kaggle free GPU notebook: not needed (LightGBM is CPU-only here).

## Expected result

- 5-fold CV ROC-AUC: **0.93 - 0.95** (per Phase 6 smoke test on 1% slice: ~0.93).
- Submission file: 286,571 rows × 2 columns (`id`, `Will_Buy_EV`).


In [12]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nContents:")
for p in Path.cwd().iterdir():
    print(" ", p.name)

print("\nDoes src exist?")
print((Path.cwd() / "src").exists())

if (Path.cwd() / "src").exists():
    print("\nContents of src:")
    for p in (Path.cwd() / "src").iterdir():
        print(" ", p.name)

Current working directory:
d:\Programming\kaggle-portfolio\competitions\predicting-electric-vehicle-purchases\notebooks

Contents:
  .gitkeep
  EDA.ipynb
  train.ipynb
  tune-optuna-changes.ipynb
  tune_optuna.ipynb

Does src exist?
False


In [13]:
# # Cell 1: Imports and path setup

# import os
# import sys
# from pathlib import Path

# import numpy as np
# import pandas as pd


# # ============================================================
# # PROJECT PATH
# # ============================================================

# PROJECT_ROOT = Path.cwd().parent

# if not (PROJECT_ROOT / "src").exists():
#     raise FileNotFoundError(
#         f"Could not find src/ at: {PROJECT_ROOT / 'src'}"
#     )

# sys.path.insert(0, str(PROJECT_ROOT))

# print(f"Project root: {PROJECT_ROOT}")
# print(f"src directory: {PROJECT_ROOT / 'src'}")


# # ============================================================
# # IMPORT PROJECT MODULES
# # ============================================================

# from src.config import (
#     CATEGORICAL_COLS,
#     FEATURE_COLS,
#     N_FOLDS,
#     RANDOM_SEED,
#     SAMPLE_SUBMISSION_CSV,
#     TARGET_COL,
#     TEST_CSV,
#     TRAIN_CSV,
# )

# from src.cv import make_folds
# from src.data import load_data
# from src.features_v2 import build_features
# from src.predict import make_submission
# from src.train_lgbm import DEFAULT_LGBM_PARAMS, train_lgbm
# from src import tracking

In [15]:
from pathlib import Path
import sys
import importlib.util

src_path = PROJECT_ROOT / "src"

print("=== PROJECT ===")
print(PROJECT_ROOT)

print("\n=== SRC DIRECTORY ===")
print(src_path)
print("Exists:", src_path.exists())
print("Is directory:", src_path.is_dir())

print("\n=== SRC FILES ===")
if src_path.exists():
    for p in sorted(src_path.iterdir()):
        print(f"{p.name:30} {'DIR' if p.is_dir() else 'FILE'}")

print("\n=== REQUIRED FILES ===")
for name in ["config.py", "cv.py", "data.py", "features.py", "features_v2.py", "__init__.py"]:
    p = src_path / name
    print(f"{name:20} exists={p.exists()}  path={p}")

print("\n=== PYTHON PATH ===")
for p in sys.path[:10]:
    print(p)

print("\n=== PYTHON'S src RESOLUTION ===")
spec = importlib.util.find_spec("src")
print(spec)


=== PROJECT ===
d:\Programming\kaggle-portfolio\competitions\predicting-electric-vehicle-purchases

=== SRC DIRECTORY ===
d:\Programming\kaggle-portfolio\competitions\predicting-electric-vehicle-purchases\src
Exists: True
Is directory: True

=== SRC FILES ===
__init__.py                    FILE
__pycache__                    DIR
config.py                      FILE
cv.py                          FILE
data.py                        FILE
features.py                    FILE
features_v2.py                 FILE
predict.py                     FILE
tracking.py                    FILE
train_lgbm.py                  FILE
tune_lgbm.py                   FILE
utils.py                       FILE

=== REQUIRED FILES ===
config.py            exists=True  path=d:\Programming\kaggle-portfolio\competitions\predicting-electric-vehicle-purchases\src\config.py
cv.py                exists=True  path=d:\Programming\kaggle-portfolio\competitions\predicting-electric-vehicle-purchases\src\cv.py
data.py          

In [16]:
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
# Cell 2: MLflow setup — point at /kaggle/working/mlruns/ so runs persist
# as a notebook output artifact.

MLRUNS_DIR = Path("/kaggle/working/mlruns")
if MLRUNS_DIR.exists():
    # Clean previous run to keep the directory small.
    shutil.rmtree(MLRUNS_DIR)
MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
tracking.set_tracking_uri(MLRUNS_DIR)

import mlflow
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

NameError: name 'tracking' is not defined

In [ ]:
# Cell 3: Load data + build features + CV folds

print("Loading data...")
train_raw, test_raw = load_data(TRAIN_CSV, TEST_CSV)
print(f"  train shape: {train_raw.shape}")
print(f"  test shape:  {test_raw.shape}")
print(f"  positive rate: {train_raw[TARGET_COL].mean():.4f}")

print("Building features...")
train_feat = build_features(train_raw)
test_feat = build_features(test_raw)
print(f"  feature columns: {FEATURE_COLS}")
print(f"  engineered cols present: {set(FEATURE_COLS) - set(train_raw.columns) <= set(train_feat.columns)}")

print("Building CV folds...")
folds = make_folds(train_feat[TARGET_COL].to_numpy(), n_splits=N_FOLDS, seed=RANDOM_SEED)
print(f"  fold sizes: {np.bincount(folds).tolist()}")
print(f"  fold positive rates: {[float(train_feat[TARGET_COL].to_numpy()[folds == k].mean()) for k in range(N_FOLDS)]}")

In [ ]:
# Cell 4: Train LightGBM with 5-fold CV

# Tunable hyperparameters. Start with defaults; tune later with Optuna.
params = DEFAULT_LGBM_PARAMS.copy()
params["learning_rate"] = 0.05
params["num_leaves"] = 63
params["min_data_in_leaf"] = 100
num_boost_round = 5000
early_stopping_rounds = 200

print("Training LightGBM with 5-fold CV...")
print(f"  params: {params}")
print(f"  num_boost_round: {num_boost_round}")
print(f"  early_stopping_rounds: {early_stopping_rounds}")

oof, test_pred, metrics = train_lgbm(
    train=train_feat,
    test=test_feat,
    folds=folds,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    params=params,
    num_boost_round=num_boost_round,
    early_stopping_rounds=early_stopping_rounds,
    categorical_cols=CATEGORICAL_COLS,
    tracking_enabled=True,
    run_name="kaggle_full_v1",
)

print()
print("=" * 60)
print("Training complete.")
print(f"  Mean CV AUC:  {metrics['cv_auc_mean']:.5f}")
print(f"  Std CV AUC:   {metrics['cv_auc_std']:.5f}")
print(f"  Per-fold AUC: {[f'{a:.5f}' for a in metrics['fold_aucs']]}")
print("=" * 60)

In [ ]:
# Cell 5: Build submission file

submission_path = Path("/kaggle/working/submission.csv")
make_submission(
    test_ids=test_feat["id"],
    test_pred=test_pred,
    template_path=SAMPLE_SUBMISSION_CSV,
    out_path=submission_path,
)
print(f"Submission written: {submission_path}")

# Sanity check: read it back and assert shape + id preservation.
sub = pd.read_csv(submission_path)
print(f"  shape: {sub.shape}")
print(f"  columns: {list(sub.columns)}")
print(f"  Will_Buy_EV range: [{sub['Will_Buy_EV'].min():.4f}, {sub['Will_Buy_EV'].max():.4f}]")
print(f"  Will_Buy_EV mean:  {sub['Will_Buy_EV'].mean():.4f}")
assert sub.shape[0] == 286_571
assert sub["id"].is_monotonic_increasing
assert (sub["Will_Buy_EV"] >= 0).all() and (sub["Will_Buy_EV"] <= 1).all()
print("Sanity checks passed.")

In [ ]:
# Cell 6: Zip MLflow runs so they can be downloaded as a notebook output.

import shutil
from pathlib import Path

working = Path("/kaggle/working")
mlruns_zip = working / "mlruns.zip"
if mlruns_zip.exists():
    mlruns_zip.unlink()
shutil.make_archive(
    base_name=str(working / "mlruns"),
    format="zip",
    root_dir=str(working),
    base_dir="mlruns",
)
print(f"MLflow runs zipped: {mlruns_zip}")
print(f"Size: {mlruns_zip.stat().st_size / 1024:.1f} KB")

In [ ]:
# Cell 7: Optional — view the leaderboard score locally by re-loading
# the OOF predictions and computing PR-AUC as a secondary check.
from sklearn.metrics import average_precision_score, roc_auc_score

y = train_feat[TARGET_COL].to_numpy()
oof_pr = average_precision_score(y, oof)
oof_roc = roc_auc_score(y, oof)
print(f"Pooled OOF AUC: {oof_roc:.5f}")
print(f"Pooled OOF PR-AUC: {oof_pr:.5f}")
print(f"(These match the MLflow run metrics within rounding.)")

## Next steps

1. **Submit** the resulting `submission.csv` to the Kaggle competition.
2. **Tune** the LGBM hyperparameters with Optuna (Phase 13 — separate from this notebook).
3. **Ensemble** with XGBoost and CatBoost (Phase 14 — separate notebook).
4. **Inspect** the MLflow run in the Kaggle sidebar (Outputs → mlruns.zip → download → `mlflow ui`).

For the model card and experiment log, update `docs/model_card.md` and `docs/experiment_log.md` with the Kaggle leaderboard score.